In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

path = kagglehub.dataset_download("shaistashahid/human-trust-levels-in-ai-systems")

print("Path to dataset files:", path)

/Users/romario/ML_from_scratch/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/romario/ML_from_scratch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/romario/.cache/kagglehub/datasets/shaistashahid/human-trust-levels-in-ai-systems/versions/1


In [2]:
file_path = "ai_skepticism_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "shaistashahid/human-trust-levels-in-ai-systems",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df.head()

/var/folders/9b/2fjcrmgn6cn8c66zrqxjrw780000gn/T/ipykernel_65535/2212887.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,ai_model_name,query_category,ai_confidence_percentage,response_character_count,has_cited_sources,contains_hedging_words,includes_disclaimer,answer_detail_level,respondent_age_bracket,education_level,...,urgency_level,belief_alignment_status,subject_matter_expertise,trust_score_out_of_10,performed_fact_check,fact_check_method_used,verification_duration_mins,answer_accuracy_percentage,trust_calibration_valid,user_skepticism_category
0,Claude,math_calculation,93.23,527,True,False,False,Moderate,45-54,Bachelors,...,Low,False,False,10.0,True,Google Search,10.7,80.15,True,Moderate Trust
1,Llama,recipe_cooking,84.47,581,False,False,False,Specific,55-64,PhD,...,NaN,False,True,7.9,True,Asked Expert,44.3,92.33,True,Skeptical
2,Claude,general_knowledge,69.82,484,True,True,False,Very Specific,35-44,High School,...,NaN,True,False,8.6,True,Consulted Documentation,37.5,67.32,True,Moderate Trust
3,Claude,creative_writing,79.61,73,True,True,False,Specific,45-54,Professional,...,NaN,NaN,False,8.9,True,Checked Official Source,22.7,73.12,True,Moderate Trust
4,Claude,creative_writing,67.71,146,False,True,False,Vague,55-64,Masters,...,NaN,True,True,9.0,True,Academic Paper,43.7,81.05,True,Moderate Trust


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [4]:
df.info(), df.isnull().sum(), df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ai_model_name               1000 non-null   object 
 1   query_category              1000 non-null   object 
 2   ai_confidence_percentage    1000 non-null   float64
 3   response_character_count    1000 non-null   int64  
 4   has_cited_sources           1000 non-null   bool   
 5   contains_hedging_words      1000 non-null   bool   
 6   includes_disclaimer         1000 non-null   bool   
 7   answer_detail_level         1000 non-null   object 
 8   respondent_age_bracket      1000 non-null   object 
 9   education_level             1000 non-null   object 
 10  digital_literacy_score      1000 non-null   object 
 11  ai_familiarity_level        1000 non-null   object 
 12  decision_importance         1000 non-null   object 
 13  urgency_level               779 no

(None,
 ai_model_name                   0
 query_category                  0
 ai_confidence_percentage        0
 response_character_count        0
 has_cited_sources               0
 contains_hedging_words          0
 includes_disclaimer             0
 answer_detail_level             0
 respondent_age_bracket          0
 education_level                 0
 digital_literacy_score          0
 ai_familiarity_level            0
 decision_importance             0
 urgency_level                 221
 belief_alignment_status       342
 subject_matter_expertise        0
 trust_score_out_of_10           0
 performed_fact_check            0
 fact_check_method_used        372
 verification_duration_mins      0
 answer_accuracy_percentage      0
 trust_calibration_valid         0
 user_skepticism_category        0
 dtype: int64,
 np.int64(0))

In [5]:
cat_features = [feature for feature in df.columns if df[feature].dtype != "float64" and df[feature].dtype != "int64"]
[df[feature].unique() for feature in cat_features]

[array(['Claude', 'Llama', 'Mistral', 'ChatGPT-3.5', 'Gemini', 'GPT-4'],
       dtype=object),
 array(['math_calculation', 'recipe_cooking', 'general_knowledge',
        'creative_writing', 'technical_coding', 'financial_advice',
        'medical_advice', 'current_events', 'factual_historical',
        'opinion_based', 'legal_advice', 'scientific_facts'], dtype=object),
 array([ True, False]),
 array([False,  True]),
 array([False,  True]),
 array(['Moderate', 'Specific', 'Very Specific', 'Vague'], dtype=object),
 array(['45-54', '55-64', '35-44', '18-24', '65+', '25-34'], dtype=object),
 array(['Bachelors', 'PhD', 'High School', 'Professional', 'Masters'],
       dtype=object),
 array(['Low', 'Expert', 'Medium', 'High'], dtype=object),
 array(['Intermediate', 'First Time', 'Advanced', 'Expert', 'Beginner'],
       dtype=object),
 array(['Low', 'High', 'Critical', 'Medium'], dtype=object),
 array(['Low', nan, 'High', 'Medium'], dtype=object),
 array([False, True, nan], dtype=object),
 

In [6]:
df['fact_check_method_used'] = df['fact_check_method_used'].fillna('Not used')
df['urgency_level'] = df['urgency_level'].fillna('Not defined')
df['belief_alignment_status'] = df['belief_alignment_status'].fillna('Not defined')

df.isnull().sum()

ai_model_name                 0
query_category                0
ai_confidence_percentage      0
response_character_count      0
has_cited_sources             0
contains_hedging_words        0
includes_disclaimer           0
answer_detail_level           0
respondent_age_bracket        0
education_level               0
digital_literacy_score        0
ai_familiarity_level          0
decision_importance           0
urgency_level                 0
belief_alignment_status       0
subject_matter_expertise      0
trust_score_out_of_10         0
performed_fact_check          0
fact_check_method_used        0
verification_duration_mins    0
answer_accuracy_percentage    0
trust_calibration_valid       0
user_skepticism_category      0
dtype: int64

In [7]:
X = df.drop(columns=['user_skepticism_category']).copy()
y = df['user_skepticism_category'].copy()

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = [
    col for col in X_train_raw.columns
    if X_train_raw[col].dtype == 'object' or X_train_raw[col].dtype == 'bool'
]
num_features = [col for col in X_train_raw.columns if col not in cat_features]

len(cat_features), len(num_features)

(17, 5)

In [8]:
df['ai_confidence_percentage'].describe()

count    1000.000000
mean       72.006790
std        14.663334
min        40.520000
25%        60.570000
50%        74.235000
75%        83.862500
max        98.660000
Name: ai_confidence_percentage, dtype: float64

In [9]:
df['response_character_count'].describe()

count    1000.000000
mean      429.022000
std       221.644937
min        50.000000
25%       239.500000
50%       423.000000
75%       629.000000
max       799.000000
Name: response_character_count, dtype: float64

In [10]:
df['verification_duration_mins'].describe()

count    1000.000000
mean       14.840000
std        15.182336
min         0.000000
25%         0.000000
50%        11.050000
75%        28.200000
max        45.000000
Name: verification_duration_mins, dtype: float64

In [11]:
df['answer_accuracy_percentage'].describe()

count    1000.000000
mean       69.982020
std        18.176782
min        19.420000
25%        58.570000
50%        72.080000
75%        83.140000
max       100.000000
Name: answer_accuracy_percentage, dtype: float64

In [12]:
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

feature_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
    X_test[col] = X_test[col].astype(str).map(mapping).fillna(-1).astype(int)
    feature_encoders[col] = le

scaler = StandardScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(y_train_raw.astype(str))
y_test = y_test_raw.astype(str).map(
    {cls: idx for idx, cls in enumerate(target_encoder.classes_)}
).fillna(-1).astype(int)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 22), (200, 22), (800,), (200,))

# Model training

In [13]:
import importlib
import decision_tree

importlib.reload(decision_tree)
from decision_tree import DecisionTree

from sklearn.metrics import accuracy_score



In [14]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy() if hasattr(y_train, 'to_numpy') else y_train
y_test = y_test.to_numpy() if hasattr(y_test, 'to_numpy') else y_test

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 22), (200, 22), (800,), (200,))

In [15]:
clf = DecisionTree(max_depth=12, impurity_method='entropy')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(accuracy_score(y_test, y_pred))

clf.print_tree()

1.0
|--- Feature 16 <= 0.009648661798425296
|   |--- Feature 17 <= 0.0
|   |   |--- Feature 16 <= -1.1172023511562916
|   |   |   |--- class: 3.0
|   |   |--- Feature 16 > -1.1172023511562916
|   |   |   |--- class: 2.0
|   |--- Feature 17 > 0.0
|   |   |--- Feature 16 <= -1.6806278576336502
|   |   |   |--- class: 1.0
|   |   |--- Feature 16 > -1.6806278576336502
|   |   |   |--- class: 3.0
|--- Feature 16 > 0.009648661798425296
|   |--- Feature 17 <= 0.0
|   |   |--- class: 0.0
|   |--- Feature 17 > 0.0
|   |   |--- class: 2.0


In [16]:
clf = DecisionTree(max_depth=12, impurity_method='gini')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(accuracy_score(y_test, y_pred))

clf.print_tree()

1.0
|--- Feature 16 <= 0.009648661798425296
|   |--- Feature 17 <= 0.0
|   |   |--- Feature 16 <= -1.1172023511562916
|   |   |   |--- class: 3.0
|   |   |--- Feature 16 > -1.1172023511562916
|   |   |   |--- class: 2.0
|   |--- Feature 17 > 0.0
|   |   |--- Feature 16 <= -1.6806278576336502
|   |   |   |--- class: 1.0
|   |   |--- Feature 16 > -1.6806278576336502
|   |   |   |--- class: 3.0
|--- Feature 16 > 0.009648661798425296
|   |--- Feature 17 <= 0.0
|   |   |--- class: 0.0
|   |--- Feature 17 > 0.0
|   |   |--- class: 2.0


In [18]:
import numpy as np

In [19]:
X_train_drop = np.delete(X_train, [16, 17], axis=1)
X_test_drop = np.delete(X_test, [16, 17], axis=1)

clf_entropy_drop = DecisionTree(max_depth=12, impurity_method='entropy')
clf_entropy_drop.fit(X_train_drop, y_train)

y_pred_entropy_drop = clf_entropy_drop.predict(X_test_drop)
print('Entropy without features 16 & 17:', accuracy_score(y_test, y_pred_entropy_drop))

clf_entropy_drop.print_tree()

Entropy without features 16 & 17: 0.77
|--- Feature 17 <= -0.9856394488775397
|   |--- Feature 2 <= -0.06652268407117654
|   |   |--- Feature 14 <= 1.0
|   |   |   |--- Feature 19 <= 0.0
|   |   |   |   |--- Feature 18 <= -0.8669197490333197
|   |   |   |   |   |--- Feature 6 <= 0.0
|   |   |   |   |   |   |--- Feature 7 <= 2.0
|   |   |   |   |   |   |   |--- Feature 3 <= -1.5945239695649758
|   |   |   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |   |   |--- Feature 3 > -1.5945239695649758
|   |   |   |   |   |   |   |   |--- class: 2.0
|   |   |   |   |   |   |--- Feature 7 > 2.0
|   |   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |--- Feature 6 > 0.0
|   |   |   |   |   |   |--- class: 2.0
|   |   |   |   |--- Feature 18 > -0.8669197490333197
|   |   |   |   |   |--- class: 0.0
|   |   |   |--- Feature 19 > 0.0
|   |   |   |   |--- Feature 18 <= -0.8492845873474075
|   |   |   |   |   |--- Feature 0 <= 1.0
|   |   |   |   |   |   |--- class: 3.0
|   |   |   

In [20]:
clf_gini_drop = DecisionTree(max_depth=12, impurity_method='gini')
clf_gini_drop.fit(X_train_drop, y_train)

y_pred_gini_drop = clf_gini_drop.predict(X_test_drop)
print('Gini without features 16 & 17:', accuracy_score(y_test, y_pred_gini_drop))

clf_gini_drop.print_tree()

Gini without features 16 & 17: 0.77
|--- Feature 17 <= -0.9856394488775397
|   |--- Feature 2 <= -0.06652268407117654
|   |   |--- Feature 14 <= 1.0
|   |   |   |--- Feature 2 <= -1.7444862368045992
|   |   |   |   |--- class: 3.0
|   |   |   |--- Feature 2 > -1.7444862368045992
|   |   |   |   |--- Feature 19 <= 0.0
|   |   |   |   |   |--- Feature 18 <= -0.8669197490333197
|   |   |   |   |   |   |--- Feature 2 <= -0.5705263885552084
|   |   |   |   |   |   |   |--- Feature 18 <= -2.3664595911385358
|   |   |   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |   |   |--- Feature 18 > -2.3664595911385358
|   |   |   |   |   |   |   |   |--- class: 2.0
|   |   |   |   |   |   |--- Feature 2 > -0.5705263885552084
|   |   |   |   |   |   |   |--- Feature 1 <= 1.0
|   |   |   |   |   |   |   |   |--- class: 0.0
|   |   |   |   |   |   |   |--- Feature 1 > 1.0
|   |   |   |   |   |   |   |   |--- class: 2.0
|   |   |   |   |   |--- Feature 18 > -0.8669197490333197
|   |   |   |   |